In [ ]:
import os
import json
import csv
import re
import time
from collections import Counter
from groq import Groq
from datasets import load_dataset
from openai import OpenAI
from utils import get_hq_ids

In [ ]:
API_KEY = "Your key here"

router_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)
groq_client = groq_client = Groq(api_key=API_KEY)

In [ ]:
model_names = {
"openai/gpt-oss-120b": "gpt-120b",
"openai/gpt-oss-20b": "gpt-20b",
"llama-3.3-70b-versatile": "llama70b",
"poolside/laguna-s-2.1:free": "laguna-s-2.1",
"google/gemma-4-26b-a4b-it:free": "gemma-4-26b", 
"nvidia/nemotron-3-super-120b-a12b:free":"nemotron-3-super",
"llama-3.1-8b-instant": "llama8b"
}

In [ ]:
MODELS = [
    ("groq", "llama-3.1-8b-instant"),
    ("groq", "llama-3.3-70b-versatile"),
    ("groq", "openai/gpt-oss-20b"),
    ("groq", "openai/gpt-oss-120b"),
    ("router","poolside/laguna-s-2.1:free"),
    ("router","google/gemma-4-26b-a4b-it:free"), 
    ("router","nvidia/nemotron-3-super-120b-a12b:free")
]

LETTERS = ["A", "B", "C", "D"]
REASONING_MODELS = ("gpt-oss", "DeepSeek", "laguna", "nemotron", "gemma")
HYBRID_MODELS = ("qwen3", "qwen2.5")

QUESTIONS_JSON_PATH = "data/selected_questions.json"
SC_CSV_PATH = "data/self_consist_mmlu_results.csv"
mmlu_results = "data/mmlu_benchmark_results.csv"

In [ ]:
treshold = 4
ABS_QUESTION_IDS = get_hq_ids(mmlu_results ,model_names,treshold)
print(len(ABS_QUESTION_IDS))

In [ ]:
from collections import Counter

K_QUERIES = 7 
SC_JSON_PATH ="data/self_consistency_mmlu.json"


sc_data = {} 

#  Helper Functions ---
def format_question(item):
    choices = "\n".join(f"{LETTERS[idx]}. {c}" for idx, c in enumerate(item["choices"]))
    return (
        f"Answer the following multiple-choice question. "
        f"Reply with ONLY the letter (A, B, C, or D) of the correct answer and nothing else, do not explain or repeat the question.\n\n"
        f"Question: {item['question']}\n{choices}\n\nAnswer:"
    )

def extract_letter(text):
    match = re.search(r"\b([ABCD])\b", text.strip().upper())
    return match.group(1) if match else None

def ask(provider, model, prompt, retries=3):
    client = groq_client if provider == "groq" else router_client
    for attempt in range(retries):
        try:
            kwargs = dict(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7, 
            )
            if any(tag in model for tag in REASONING_MODELS):
                kwargs["max_tokens"] = 2048
                kwargs["reasoning_effort"] = "low"
            elif any(tag in model for tag in HYBRID_MODELS):
                kwargs["max_tokens"] = 2048
                kwargs["reasoning_format"] = "hidden"
            else:
                kwargs["max_tokens"] = 10
                
            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limited on {model}, waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"  Error on {model}: {e}")
                return None
    return None

def save_sc_json():
    """Saves the current state of sc_data directly to a JSON file."""
    with open(SC_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(sc_data, f, indent=4)

# --- 4. Load Data ---
if not os.path.exists(QUESTIONS_JSON_PATH):
    raise FileNotFoundError(f"{QUESTIONS_JSON_PATH} not found. Please run the original script first.")

with open(QUESTIONS_JSON_PATH, "r", encoding="utf-8") as f:
    all_items = json.load(f)

# Filter items to only include our hard questions
target_items = [item for item in all_items if item["question_id"] in HARD_QUESTION_IDS]
print(f"Loaded {len(target_items)} hard questions from JSON.")

# Initialize the sc_data dictionary with a clean structure for each question
for item in target_items:
    sc_data[item["question_id"]] = {
        "correct_label": item["answer"],
        "models": {}
    }

# SC Execution Loop 
print("\nStarting self-consistency loop...")

for idx, item in enumerate(target_items):
    q_id = item["question_id"]
    prompt = format_question(item)  
    print(f"\n--- Processing {q_id} ({idx+1}/{len(target_items)}) ---")
    
    for provider, model in MODELS:
        print(f"Querying {model} ({provider}) {K_QUERIES} times...")
        responses = []
        
        for k in range(K_QUERIES):
            answer_text = ask(provider, model, prompt)
            predicted = extract_letter(answer_text or "")
            if predicted:
                responses.append(predicted)
            
            # Short sleep between queries to prevent instant rate limiting
            time.sleep(1) 
        
        # Base dictionary with all options zeroed out
        vote_dict = {"A": 0, "B": 0, "C": 0, "D": 0}

        if not responses:
            print(f"  -> {model} failed to return valid answers.")
            sc_data[q_id]["models"][model] = vote_dict
            continue
            
        # Count occurrences of A, B, C, D
        counts = Counter(responses)
        
        # Update our base dictionary with the actual counts
        for letter in vote_dict.keys():
            vote_dict[letter] = counts.get(letter, 0)
        
        print(f"  -> {model} distribution: {vote_dict}")
        
        # Save the dictionary natively into the JSON structure
        sc_data[q_id]["models"][model] = vote_dict
            
        # Save to JSON after every model finishes its K queries
        save_sc_json()
        time.sleep(2) # Buffer before switching to the next model

print("\nEvaluation complete! Results saved to", SC_JSON_PATH)